In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import uuid
from datetime import datetime, timezone

In [0]:
spark.sql("use catalog neogen")

In [0]:
spark.sql("create schema if not exists silver_schema")

In [0]:
silver_run_id=str(uuid.uuid4())
print(silver_run_id)

In [0]:
spark.sql("""create table if not exists neogen.silver_schema.process_control (
    layer string,
    entity_name string,
    last_processed_bronze_run_id string,
    last_processed_bronze_ingested_at timestamp,
    rows_merged bigint,   
    run_status string,
    silver_run_id string,
    updated_at timestamp
)using delta""")

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        # Load existing Delta table
        dt = DeltaTable.forName(spark, target_table)
        # Perform merge (upsert)
        (
            dt.alias("tgt")
              .merge(
                  df_source.alias("src"),
                  f"tgt.{join_key} = src.{join_key}"
              )
              .whenMatchedUpdateAll()      # Update all columns if matched
              .whenNotMatchedInsertAll()   # Insert new rows if not matched
              .execute()
        )
    else:
        # If table doesn’t exist, create it
        df_source.write.format("delta").saveAsTable(target_table)

    

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl_df = (
        spark.table("neogen.silver_schema.process_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl_df.collect()
    if not rows:
        return None
    return rows[0]["last_processed_bronze_ingested_at"]



In [0]:
def upsert_silver_control(
    entity_name: str,
    last_processed_bronze_run_id: str,
    last_processed_bronze_ingested_at,
    rows_merged: int,
    silver_run_id: str):

    # Define schema
    schema = """layer string,entity_name string,last_processed_bronze_run_id string,last_processed_bronze_ingested_at timestamp,rows_merged bigint,run_status string,silver_run_id string,updated_at timestamp """

    # Create single-row DataFrame with metadata
    ctrl_df = spark.createDataFrame(
        [
            {
                "layer": "silver",
                "entity_name": entity_name,
                "last_processed_bronze_run_id": last_processed_bronze_run_id,
                "last_processed_bronze_ingested_at": last_processed_bronze_ingested_at,
                "rows_merged": int(rows_merged),
                "run_status": "SUCCESS",
                "silver_run_id": silver_run_id,
                "updated_at": datetime.now(timezone.utc)
            }
        ],
        schema=schema
    )

    # Merge into Delta control table
    dt = DeltaTable.forName(spark, "neogen.silver_schema.process_control")
    (
        dt.alias("t")
          .merge(ctrl_df.alias("s"), "t.layer = s.layer AND t.entity_name = s.entity_name")
          .whenMatchedUpdate(set={
              "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
              "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
              "rows_merged": "s.rows_merged",
              "run_status": "s.run_status",
              "silver_run_id": "s.silver_run_id",
              "updated_at": "s.updated_at"
          })
          .whenNotMatchedInsertAll()
          .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table: str, entity_name: str):
    # Step 1: Get the last processed bronze ingestion timestamp
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    # Step 2: Read the bronze table
    bronze_df = spark.read.table(bronze_table)

    # Step 3: If no previous ingestion, return full bronze data
    if last_ingested_at is None:
        return bronze_df, last_ingested_at

    # Step 4: Otherwise, filter only new records since last ingestion
    return (
        bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),
        last_ingested_at
    )


In [0]:
# Step 4 - Orders incremental processing
# Read only the Bronze order rows that Silver has not processed yet.
orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "neogen.bronze_schema.orders_raw",
    "orders")

# Count the incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows to process in silver = {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows.
if orders_inc_count > 0:
    # Create a window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )
    # Start the Silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
    orders_cleaned = (
        orders_inc
        # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn(
            "order_status",
            F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status"))
        )
        # Remove formatting characters from order_amount so it can be cast to a numeric type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn(
            "order_amount",
            F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None)
            .otherwise(F.col("order_amount"))
        )
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(order_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        # Add silver run tracking ID for auditability.
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )
    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_cleaned,
        "neogen.silver_schema.orders_cleaned",
        "order_id" )
    
    
    # Apply Silver data-quality rules to the cleaned order records.
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )

    # Keep only valid order rows for the transformed Silver table.
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")

    # Send invalid rows to the quarantine dataset for manual review.
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "neogen.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad order rows to the quarantine table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "neogen.silver_schema.orders_quarantine"
    )

    # Track the latest ingestion timestamp and run_id from Bronze for control updates.
    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    # Update Silver control table with metadata for this run.
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count(),silver_run_id)
else:
    print("No new orders Bronze rows for Silver.")
    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count,silver_run_id
    )

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver has not processed yet.
products_inc, last_products_ingested_at = get_incremental_bronze(
    "neogen.bronze_schema.products_raw",
    "products"
)

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products rows to process in silver = {products_inc_count}")

if products_inc.count() > 0:
    # Create a window that keeps the latest product record for each product_id.
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
    products_cleaned = (
        products_inc
        # Standardize product_name by trimming spaces and converting text to uppercase.
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
        .withColumn(
            "product_name",
            F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name"))
        )
        # Standardize category values to consistent labels.
        .withColumn(
            "category",
            F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRONICS"), "ELECTRONICS")
             .when(F.upper(F.trim(F.col("category"))) == "FITNESS", "FITNESS")
             .when(F.upper(F.trim(F.col("category"))) == "LIFESTYLE", "LIFESTYLE")
             .otherwise(F.upper(F.trim(F.col("category"))))
        )

        # Start cleaning the product price field before converting it to numeric.
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price", F.regexp_replace(F.col("price"), "[$,]", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), "[^0-9.]", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), "\\s+", ""))
        .withColumn("price", F.expr("try_cast(price as double)"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(product_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )
    
    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        products_cleaned,
        "neogen.silver_schema.products_cleaned",
        "product_id"
    )

    # Apply Silver data-quality rules to the cleaned product records.
    products_validated = (
        products_cleaned
            .withColumn(
                "to_be_verified_by_products_team",
                F.when(F.col("product_name").isNull(), "verify_product_name")
                .when(F.col("category").isNull(), "verify_category")
                .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
                .otherwise("No Issues")
            )
            .withColumn(
                "check_product_price",
                F.when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price")
                .otherwise("valid_price")
            )
    )

    # Keep only valid product rows for the transformed Silver table.
    products_good = (
        products_validated
        .filter(
            (F.col("to_be_verified_by_products_team") == "No Issues") &
            (F.col("check_product_price") == "valid_price")
        )
        .drop("price_raw")
    )

    # Send invalid product rows to the quarantine dataset for manual review.
    products_bad = (
        products_validated
        .filter(
            (F.col("to_be_verified_by_products_team") != "No Issues") |
            (F.col("check_product_price") == "invalid_price")
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        products_good,
        "neogen.silver_schema.products_transformed",
        "product_id"
    )

    # Append bad product rows to the quarantine table instead of losing them.
    products_bad.write.format("delta").mode("append").saveAsTable(
        "neogen.silver_schema.products_quarantine"
    )

    # Track the latest ingestion timestamp and run_id from Bronze for control updates.
    mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    upsert_silver_control("products", mx_run, mx_ingested, products_good.count(),silver_run_id)

else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count,silver_run_id
    )

In [0]:
# Step 6 - Payments incremental processing
# Read only the Bronze payment rows that Silver has not processed yet.
payments_inc, last_payments_ingested_at = get_incremental_bronze(
    "neogen.bronze_schema.payments_raw",
    "payments"
)
print("Payments last processed Bronze ingested_at =", last_payments_ingested_at)

# Count the incremental payment rows entering Silver in this run.
payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc.count() > 0:
    # Create a window that keeps the latest payment record for each payment_id.
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )
    # Start the Silver payment-cleaning pipeline. This block standardizes and deduplicates raw payment records.
    payments_cleaned = (
        payments_inc
        # Standardize payment_status by trimming spaces and converting text to uppercase.
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
        .withColumn(
            "payment_status",
            F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status"))
        )
        # Start cleaning the payment amount field before converting it to numeric.
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), ",", "."))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\s+", ""))
        .withColumn(
            "paid_amount",
            F.when(F.trim(F.col("paid_amount")).isin("N/A", "NULL", "??", ""), None)
            .otherwise(F.col("paid_amount"))
        )
        .withColumn("paid_amount", F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", F.to_timestamp("processed_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(payment_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        payments_cleaned,
        "neogen.silver_schema.payments_cleaned",
        "payment_id"
    )

    # Apply Silver data-quality rules to the cleaned payment records.
    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull(), "verify_payment_status")
            .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_paid_amount",
            F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
    )

    # Keep only valid payment rows for the transformed Silver table.
    payments_good = payments_validated.filter(F.col("to_be_verified_by_payments_team") == "No Issues")

    # Send invalid payment rows to the quarantine dataset for manual review.
    payments_bad = (
        payments_validated
        .filter(F.col("to_be_verified_by_payments_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        payments_good,
        "neogen.silver_schema.payments_transformed",
        "payment_id"
    )

    # Append bad payment rows to the quarantine table instead of losing them.
    payments_bad.write.format("delta").mode("append").saveAsTable(
        "neogen.silver_schema.payments_quarantine"
    )
    # Track the latest ingestion timestamp and run_id from Bronze for control updates.
    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    # Update Silver control table with metadata for this run.
    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count(),silver_run_id)

else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count,silver_run_id
    )

In [0]:
print("Products transformed count :", spark.sql(
    "SELECT COUNT(*) FROM neogen.silver_schema.products_transformed"
).collect()[0][0])

print("Orders transformed count :", spark.sql(
    "SELECT COUNT(*) FROM neogen.silver_schema.orders_transformed"
).collect()[0][0])

print("Payments transformed count :", spark.sql(
    "SELECT COUNT(*) FROM neogen.silver_schema.payments_transformed"
).collect()[0][0])

display(
    spark.table("neogen.silver_schema.process_control")
    .orderBy("entity_name")
)
